# Hy3 API - 推理模式示例

本示例演示 Hy3 API 的推理模式（reasoning_effort）开关，对比开启和关闭思考过程的差异。

In [ ]:
import time
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(
    api_key=os.getenv("API_KEY", "EMPTY"),
    base_url=os.getenv("BASE_URL", "http://127.0.0.1:8000/v1")
)

## 推理模式参数

Hy3 提供三种推理模式，通过 `reasoning_effort` 参数控制：

| 模式 | 参数值        | 适用场景 |
|:---|:-----------|:---|
| 直接回复 | `"low"`    | 日常对话、简单问答（默认） |
| 轻度思考 | `"medium"` | 一般推理任务 |
| 深度思维链 | `"high"`   | 数学计算、逻辑推理、编程 |

In [ ]:
def run_reasoning_test(question, reasoning_effort):
    start_time = time.time()

    response = client.chat.completions.create(
        model="hy3",
        messages=[
            {"role": "user", "content": question},
        ],
        temperature=0.9,
        top_p=1.0,
        extra_body = {
        "thinking": {"type": "enabled"},
        "reasoning_effort": reasoning_effort
    },
    )

    elapsed = time.time() - start_time

    return {
        "reasoning_effort": reasoning_effort,
        "content": response.choices[0].message.content,
        "time": elapsed,
        "usage": response.usage,
    }

## 测试用例 1：简单问题

In [ ]:
question = "2+2等于多少？"

result_low = run_reasoning_test(question, "low")
time.sleep(2)
result_high = run_reasoning_test(question, "high")
time.sleep(2)

print(f"【low 模式】")
print(f"  耗时: {result_low['time']:.4f}s")
print(f"  Token用量: {result_low['usage'].total_tokens}")
print(f"  回复: {result_low['content']}")

print(f"\n【high 模式】")
print(f"  耗时: {result_high['time']:.4f}s")
print(f"  Token用量: {result_high['usage'].total_tokens}")
print(f"  回复: {result_high['content']}")

## 测试用例 2：数学推理

In [ ]:
question = "一个水池有两个进水管和一个出水管。单独开甲管6小时注满，单独开乙管4小时注满，单独开丙管3小时放完。现在三管同时打开，几小时可以注满水池？"

result_low = run_reasoning_test(question, "low")
time.sleep(2)
result_high = run_reasoning_test(question, "high")
time.sleep(2)

print(f"【low 模式】")
print(f"  耗时: {result_low['time']:.4f}s")
print(f"  回复: {result_low['content'][:150]}...")

print(f"\n【high 模式】")
print(f"  耗时: {result_high['time']:.4f}s")
print(f"  回复: {result_high['content']}")

## 测试用例 3：逻辑推理

In [ ]:
question = "有A、B、C、D四个人，他们分别来自北京、上海、广州、深圳。已知：1) A不是北京人；2) B既不是上海人也不是北京人；3) 来自广州的不是C；4) D是深圳人。请问每个人分别来自哪里？"

result_low = run_reasoning_test(question, "low")
time.sleep(2)
result_high = run_reasoning_test(question, "high")
time.sleep(2)

print(f"【low 模式】")
print(f"  耗时: {result_low['time']:.4f}s")
print(f"  回复: {result_low['content']}")

print(f"\n【high 模式】")
print(f"  耗时: {result_high['time']:.4f}s")
print(f"  回复: {result_high['content']}")

## 对比分析

| 指标 | low     | high |
|:---|:--------|:---|
| **响应速度** | 快       | 慢（+30-50%） |
| **Token 消耗** | 少       | 多（+50-100%） |
| **准确性** | 一般      | 高 |
| **思考过程** | 无       | 有（Chain of Thought） |
| **适用场景** | 简单问答、闲聊 | 数学、编程、逻辑推理 |

## 关键要点

1. **默认模式**：`low` 是默认值，直接响应，速度最快
2. **思考过程**：开启 `high` 模式后，模型会输出思考过程（Chain of Thought）
3. **性能代价**：思考模式会增加响应时间和 Token 消耗
4. **选择策略**：
   - 简单问答 → `low`
   - 一般推理 → `medi`
   - 复杂任务 → `high`